# Cosine Similarity - word

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load a small pretrained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Words (or short texts)
words = ["cat", "cats", "kitten", "dog", "dogs", "milk", "silk"]

# Convert to embeddings
embeddings = model.encode(words, convert_to_tensor=False)

# Compute cosine similarity
cos_sim = cosine_similarity(embeddings)

# Display results as a DataFrame
df = pd.DataFrame(cos_sim, index=words, columns=words)
print(df.round(3))


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


          cat   cats  kitten    dog   dogs   milk   silk
cat     1.000  0.870   0.788  0.661  0.575  0.342  0.176
cats    0.870  1.000   0.743  0.567  0.715  0.363  0.183
kitten  0.788  0.743   1.000  0.521  0.453  0.311  0.254
dog     0.661  0.567   0.521  1.000  0.824  0.371  0.274
dogs    0.575  0.715   0.453  0.824  1.000  0.391  0.262
milk    0.342  0.363   0.311  0.371  0.391  1.000  0.365
silk    0.176  0.183   0.254  0.274  0.262  0.365  1.000


# Cosine Similarity - Sentense

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pretrained sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Query and candidates
query = "Can you give me best investing options?"
sent1 = "A kitten likes to drink milk."
sent2 = "Stock prices increased yesterday."

# Encode
embeddings = model.encode([query, sent1, sent2], convert_to_tensor=False)

# Compute cosine similarity of query vs each candidate
sim_query_sent1 = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
sim_query_sent2 = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]

print("Query:", query)
print("Sentence 1:", sent1, "→ Similarity:", round(sim_query_sent1, 3))
print("Sentence 2:", sent2, "→ Similarity:", round(sim_query_sent2, 3))

if sim_query_sent1 > sim_query_sent2:
    print("\nSentence 1 is more similar to the query.")
else:
    print("\nSentence 2 is more similar to the query.")


Query: Can you give me best investing options?
Sentence 1: A kitten likes to drink milk. → Similarity: 0.011
Sentence 2: Stock prices increased yesterday. → Similarity: 0.238

Sentence 2 is more similar to the query.


# ChromaDB

In [5]:
import chromadb

In [6]:
chroma_client = chromadb.Client()

In [7]:
collection = chroma_client.create_collection(name="my_collection")

In [8]:
collection.add(
    documents=[
        "Oranges are very sweet & healthy",
        "Orange is bright colour"
    ],
    ids=["id1", "id2"]
)

In [9]:
collection.peek()

{'ids': ['id1', 'id2'],
 'embeddings': array([[ 1.85966324e-02, -1.66588649e-02,  5.93799748e-04,
          1.28125146e-01, -5.61542902e-03,  2.09736899e-02,
          6.02619350e-02, -3.03189736e-02,  2.58360077e-02,
         -3.10340188e-02,  2.54585464e-02,  3.52508761e-02,
         -4.68438193e-02, -5.59613667e-02,  5.09437509e-02,
          8.19939226e-02,  7.51795024e-02, -5.23920730e-02,
         -1.01085663e-01, -9.20305494e-03, -1.29376380e-02,
         -7.23995864e-02,  4.86832596e-02,  6.45977333e-02,
         -5.21541946e-02, -2.70127524e-02,  5.33109419e-02,
         -2.50075981e-02, -3.43400463e-02, -3.59926149e-02,
         -5.32587171e-02, -1.83577687e-02,  9.49271247e-02,
          5.94063709e-03, -8.80596712e-02, -9.77594331e-02,
          1.02583215e-01, -7.25067705e-02, -3.09198797e-02,
         -2.53512594e-03, -4.77523776e-03,  4.32436876e-02,
          2.31515965e-03,  4.84592989e-02, -2.46634688e-02,
         -9.61348531e-04, -5.51934317e-02, -5.24972193e-02,
  

In [11]:
results = collection.query(
    query_texts=["Life is colourfull!"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

results

{'ids': [['id2', 'id1']],
 'embeddings': None,
 'documents': [['Orange is bright colour',
   'Oranges are very sweet & healthy']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[1.2615089416503906, 1.6319520473480225]]}

# ChromaDB - PDF document with PersistentClient

In [27]:
# pip install pypdf sentence-transformers chromadb

import chromadb
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

# 1. Read PDF text
reader = PdfReader("AIML.pdf")
pages = [page.extract_text() for page in reader.pages]
full_text = " ".join(pages)

# 2. Simple chunking
chunks = [full_text[i:i+500] for i in range(0, len(full_text), 500)]

# 3. Embed chunks
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks).tolist()

# 4. Store in Chroma
# Inline Memory
#client = chromadb.Client()
client = chromadb.PersistentClient(path="./vector_db")
#Persistent Client
coll = client.create_collection(name="stocks")

coll.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"source": "AIML.pdf"} for _ in chunks]
)

print("Stored", len(chunks), "chunks from AIML.pdf")


Stored 13 chunks from AIML.pdf


In [24]:
query = "Can you give me oops concepts"
res = coll.query(query_texts=[query], n_results=3)

top_doc = res["documents"][0][0]
top_dist = res["distances"][0][0]

print("Best Match:\n")
print(top_doc)   # full text of the document
print(f"\nDistance: {top_dist:.4f}")

Best Match:

es Concept 7: File & Data Processor 
If
If-else
If-elif-else
Nested-if
for
while
Nested loops
Break
ContinueConcept 8: Control Flow 
 Built-in Functions
User Defined Functions
Function Arguments
Function Return Values Concept 9: Functions 
Learn how to handle Python exceptions and raise exceptions.
Understand Try, except, else, finally, and creating custom exceptions.Concept 10: Exception Handling 
Module Introduction
Package Management
Create Python Packages
Understand how you can write modular

Distance: 1.1546
